In [111]:
states = ["Code", "Test", "Dependency", "CI/Config"] 

actions = [
    "Inspect pipeline logs",
    "Inspect failed pipeline stage",
    "Compare with last successful run",
    "Inspect recent code/test changes",
    "Inspect dependency changes",
    "Inspect CI/environment configuration",
    "Search previous incidents/runbooks",
    "Inspect Docker image/cache changes",
    "Inspect hosting environment",
    "Inspect server/session state",
    "Escalate to human"
]


priors = {
    "Code": 0.1770,
    "Test": 0.3934,
    "Dependency": 0.2984,
    "CI/Config": 0.1311
}

likelihoods = {
    "Build": {
        "Code": 0.574,
        "Test": 0.358,
        "Dependency": 0.429,
        "CI/Config": 0.275
    },
    "Test": {
        "Code": 0.185,
        "Test": 0.533,
        "Dependency": 0.132,
        "CI/Config": 0.100
    },
    "Setup / Dependency": {
        "Code": 0.074,
        "Test": 0.000,
        "Dependency": 0.176,
        "CI/Config": 0.125
    },
    "Quality / Analysis": {
        "Code": 0.019,
        "Test": 0.058,
        "Dependency": 0.055,
        "CI/Config": 0.050
    },
    "Deploy / Publish": {
        "Code": 0.000,
        "Test": 0.000,
        "Dependency": 0.033,
        "CI/Config": 0.050
    },
    "Ambiguous": {
        "Code": 0.074,
        "Test": 0.033,
        "Dependency": 0.077,
        "CI/Config": 0.150
    },
    "Unknown": {
        "Code": 0.074,
        "Test": 0.017,
        "Dependency": 0.099,
        "CI/Config": 0.250
    }
}

In [112]:
def calculate_posterior(failed_stage):
    stage_likelihoods = likelihoods[failed_stage]

    unnormalized = {}

    for state in states:
        unnormalized[state] = (
            priors[state] * stage_likelihoods[state]
        )

    evidence_probability = sum(unnormalized.values())

    posterior = {}

    for state in states:
        posterior[state] = (
            unnormalized[state] / evidence_probability
        )

    return posterior

In [113]:
posterior = calculate_posterior("Build")

for state, probability in posterior.items():
    print(f"{state}: {probability:.3%}")

Code: 24.993%
Test: 34.646%
Dependency: 31.492%
CI/Config: 8.869%


In [114]:
CONFIDENCE_THRESHOLD = 0.90


def should_stop(posterior):
    best_state = max(posterior, key=posterior.get)
    best_probability = posterior[best_state]

    if best_probability >= CONFIDENCE_THRESHOLD:
        return True, best_state

    return False, best_state

In [115]:
posterior = calculate_posterior("Build")

stop, best_state = should_stop(posterior)

print("Best hypothesis:", best_state)
print("Stop diagnosis:", stop)

Best hypothesis: Test
Stop diagnosis: False


In [116]:
actions = [
    {
        "name": "Inspect pipeline logs",
        "tags": ["all"],
        "cost": 1,
        "outcomes": [
            "Relevant error found",
            "No clear error found"
        ]
    },
    {
        "name": "Inspect failed pipeline stage",
        "tags": ["all"],
        "cost": 2,
        "outcomes": [
            "Specific failed stage identified",
            "Failure remains ambiguous"
        ]
    },
    {
        "name": "Compare with last successful run",
        "tags": ["all"],
        "cost": 2,
        "outcomes": [
            "Significant difference found",
            "No significant difference found"
        ]
    },
    {
        "name": "Inspect recent code/test changes",
        "tags": ["Build", "Test", "Quality / Analysis"],
        "cost": 1,
        "outcomes": [
            "Relevant code/test change found",
            "No relevant change found"
        ]
    },
    {
        "name": "Inspect dependency changes",
        "tags": ["Build", "Setup / Dependency"],
        "cost": 1,
        "outcomes": [
            "Dependency change found",
            "No dependency change found"
        ]
    },
    {
        "name": "Inspect CI/environment configuration",
        "tags": [
            "Build",
            "Test",
            "Setup / Dependency",
            "Quality / Analysis",
            "Deploy / Publish"
        ],
        "cost": 2.5,
        "outcomes": [
            "Configuration/environment issue found",
            "No configuration/environment issue found"
        ]
    },
    {
        "name": "Search previous incidents/runbooks",
        "tags": ["all"],
        "cost": 1,
        "outcomes": [
            "Similar incident found",
            "No similar incident found"
        ]
    },
    {
        "name": "Inspect Docker image/cache changes",
        "tags": ["Build", "Deploy / Publish"],
        "cost": 1.5,
        "outcomes": [
            "Docker/image difference found",
            "No Docker/image difference found"
        ]
    },
    {
        "name": "Inspect hosting environment",
        "tags": ["Deploy / Publish"],
        "cost": 2,
        "outcomes": [
            "Infrastructure issue found",
            "No infrastructure issue found"
        ]
    },
    {
        "name": "Inspect server/session state",
        "tags": ["Build", "Deploy / Publish"],
        "cost": 1.5,
        "outcomes": [
            "Server/session issue found",
            "No server/session issue found"
        ]
    }
]

import math

def entropy(probabilities):
    return -sum(
        p * math.log2(p)
        for p in probabilities
        if p > 0
    )


def calculate_eig(current_posterior, outcome_model):
    current_entropy = entropy(current_posterior.values())

    eig = 0

    for outcome, probabilities in outcome_model.items():

        # P(outcome | E, action)
        outcome_probability = sum(
            current_posterior[state] * probabilities[state]
            for state in states
        )

        # P(H | E, outcome, action)
        outcome_posterior = {}

        for state in states:
            numerator = (
                current_posterior[state]
                * probabilities[state]
            )

            outcome_posterior[state] = (
                numerator / outcome_probability
            )

        # Information gained from this outcome
        outcome_entropy = entropy(outcome_posterior.values())

        information_gain = (
            current_entropy - outcome_entropy
        )

        # Expected contribution of this outcome
        eig += outcome_probability * information_gain

    return eig

def rank_actions(posterior, candidate_actions, outcome_models):
    eig_known = []
    eig_unknown = []

    for action in candidate_actions:
        name = action["name"]
        cost = action["cost"]

        if name in outcome_models:
            eig = calculate_eig(
                posterior,
                outcome_models[name]
            )

            score = eig / cost

            eig_known.append({
                "name": name,
                "cost": cost,
                "eig": eig,
                "score": score
            })

        else:
            eig_unknown.append({
                "name": name,
                "cost": cost,
                "eig": None,
                "score": None
            })

    # EIG-known: highest information per cost first
    eig_known.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    # EIG-unknown: cheapest first
    eig_unknown.sort(
        key=lambda x: x["cost"]
    )

    return eig_known, eig_unknown


def get_candidate_actions(failed_stage, actions):
    candidate_actions = []

    for action in actions:
        if "all" in action["tags"] or failed_stage in action["tags"]:
            candidate_actions.append(action)

    return candidate_actions


def decide_next_step(
    posterior,
    candidate_actions,
    outcome_models
):
    # 1. Check stopping criterion
    stop, best_state = should_stop(posterior)

    if stop:
        return {
            "decision": "REPORT",
            "root_cause": best_state
        }

    # 2. Rank remaining actions
    eig_known, eig_unknown = rank_actions(
        posterior,
        candidate_actions,
        outcome_models
    )

    # 3. Prefer EIG-known actions
    if eig_known:
        return {
            "decision": "RECOMMEND",
            "action": eig_known[0]
        }

    # 4. Fall back to unknown-EIG actions
    if eig_unknown:
        return {
            "decision": "RECOMMEND",
            "action": eig_unknown[0]
        }

    # 5. No actions remain and confidence is insufficient
    return {
        "decision": "ESCALATE"
    }


def update_posterior_with_outcome(
    current_posterior,
    outcome_probabilities
):
    # P(o | E, a)
    outcome_probability = sum(
        current_posterior[state] * outcome_probabilities[state]
        for state in states
    )

    new_posterior = {}

    for state in states:
        numerator = (
            current_posterior[state]
            * outcome_probabilities[state]
        )

        new_posterior[state] = (
            numerator / outcome_probability
        )

    return new_posterior

def run_diagnosis(failed_stage, outcomes):
    posterior = calculate_posterior(failed_stage)

    remaining_actions = get_candidate_actions(
        failed_stage,
        actions
    ).copy()

    for outcome in outcomes:

        decision = decide_next_step(
            posterior,
            remaining_actions,
            outcome_models
        )

        print("\nDecision:", decision)

        if decision["decision"] == "REPORT":
            print("Root cause:", decision["root_cause"])
            return decision

        if decision["decision"] == "ESCALATE":
            print("Escalating to human.")
            return decision

        action = decision["action"]
        action_name = action["name"]

        print("Performing:", action_name)

        # Remove completed action
        remaining_actions = [
            a for a in remaining_actions
            if a["name"] != action_name
        ]

        # Simulated outcome
        outcome_probabilities = (
            outcome_models[action_name][outcome]
        )

        posterior = update_posterior_with_outcome(
            posterior,
            outcome_probabilities
        )

        print("Updated posterior:")
        for state, probability in posterior.items():
            print(f"{state}: {probability:.3%}")

    return posterior

In [117]:
outcome_models = {
    "Inspect recent code/test changes": {
        "Relevant change found": {
            "Code": 0.70,
            "Test": 0.30,
            "Dependency": 0.20,
            "CI/Config": 0.10,
        },
        "No relevant change found": {
            "Code": 0.30,
            "Test": 0.70,
            "Dependency": 0.80,
            "CI/Config": 0.90,
        },
    }
}


In [118]:
failed_stage = "Build"

def test_v1():
    # --------------------------------------------------
    # 1. Test Bayesian posterior
    # --------------------------------------------------
    posterior = calculate_posterior("Build")

    assert abs(sum(posterior.values()) - 1.0) < 1e-9
    print("✓ Posterior calculation")


    # --------------------------------------------------
    # 2. Test posterior update
    # --------------------------------------------------
    new_posterior = update_posterior_with_outcome(
        posterior,
        outcome_models["Inspect recent code/test changes"]
            ["Relevant change found"]
    )

    assert abs(sum(new_posterior.values()) - 1.0) < 1e-9
    assert new_posterior["Code"] > posterior["Code"]

    print("✓ Posterior update")


    # --------------------------------------------------
    # 3. Test EIG
    # --------------------------------------------------
    eig = calculate_eig(
        posterior,
        outcome_models["Inspect recent code/test changes"]
    )

    assert eig >= 0
    print("✓ EIG calculation")


    # --------------------------------------------------
    # 4. Test tag filtering
    # --------------------------------------------------
    candidate_actions = get_candidate_actions(
        "Build",
        actions
    )

    for action in candidate_actions:
        assert (
            "all" in action["tags"]
            or "Build" in action["tags"]
        )

    print("✓ Action filtering")


    # --------------------------------------------------
    # 5. Test action ranking
    # --------------------------------------------------
    eig_known, eig_unknown = rank_actions(
        posterior,
        candidate_actions,
        outcome_models
    )

    assert any(
        action["name"] == "Inspect recent code/test changes"
        for action in eig_known
    )

    for action in eig_unknown:
        assert action["eig"] is None

    print("✓ Action ranking")


    # --------------------------------------------------
    # 6. Test recommendation
    # --------------------------------------------------
    decision = decide_next_step(
        posterior,
        candidate_actions,
        outcome_models
    )

    assert decision["decision"] == "RECOMMEND"

    print("✓ Recommendation")


    # --------------------------------------------------
    # 7. Test reporting
    # --------------------------------------------------
    high_confidence = {
        "Code": 0.95,
        "Test": 0.02,
        "Dependency": 0.02,
        "CI/Config": 0.01
    }

    decision = decide_next_step(
        high_confidence,
        candidate_actions,
        outcome_models
    )

    assert decision["decision"] == "REPORT"
    assert decision["root_cause"] == "Code"

    print("✓ Reporting")


    # --------------------------------------------------
    # 8. Test escalation
    # --------------------------------------------------
    low_confidence = {
        "Code": 0.40,
        "Test": 0.30,
        "Dependency": 0.20,
        "CI/Config": 0.10
    }

    decision = decide_next_step(
        low_confidence,
        [],
        outcome_models
    )

    assert decision["decision"] == "ESCALATE"

    print("✓ Escalation")


    print("\nAll V1 tests passed!")

In [119]:
test_v1()

✓ Posterior calculation
✓ Posterior update
✓ EIG calculation
✓ Action filtering
✓ Action ranking
✓ Recommendation
✓ Reporting
✓ Escalation

All V1 tests passed!
